# Notebook 01: EDA and Problem Framing

## Executive framing
This project focuses on **incremental impact** for campaign targeting, not raw conversion prediction.

Business question: **Which customers should receive an email campaign to maximize incremental conversions and incremental revenue?**

## Critical modeling definition (explicit)
The original Hillstrom setup is typically 3-arm:
- `No E-Mail`
- `Mens E-Mail`
- `Womens E-Mail`

For this project, we intentionally collapse to binary treatment:
- **treated = any email** (`Mens E-Mail` or `Womens E-Mail`)
- **control = no email** (`No E-Mail`)

This is a modeling simplification for binary uplift methods.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

if Path.cwd().name == 'notebooks':
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import load_hillstrom_data, basic_cleaning, prepare_features
from src.utils import set_seed, ensure_output_dirs, save_plot, standardized_mean_difference

SEED = 42
set_seed(SEED)
OUT_DIRS = ensure_output_dirs(PROJECT_ROOT)
DATA_PATH = PROJECT_ROOT / 'data' / 'hillstrom.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data path: {DATA_PATH}')
print(f'Figures dir: {OUT_DIRS["figures"]}')

## 1) Data loading

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError('Expected dataset at data/hillstrom.csv')

raw_df = load_hillstrom_data(str(DATA_PATH))
df = basic_cleaning(raw_df)

print('Shape:', df.shape)
display(df.head())

## 2) Schema and quality checks
We explicitly identify treatment, visit, conversion, and spend columns.

In [ ]:
required_cols = ['segment', 'treatment', 'visit', 'conversion', 'spend']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

schema = pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str)})
missingness = (df.isna().mean() * 100).rename('missing_pct').sort_values(ascending=False).to_frame()
summary = df[['visit', 'conversion', 'spend', 'treatment']].describe().T

display(schema)
display(missingness)
display(summary)

## 3) Initial exploratory analysis

In [ ]:
numeric_cols = [c for c in ['recency', 'history', 'visit', 'conversion', 'spend'] if c in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], bins=40, ax=axes[i], color='#2E86AB')
    axes[i].set_title(f'{col} distribution')
for j in range(len(numeric_cols), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb01_numeric_distributions.png')
plt.show()

In [ ]:
cat_cols = [c for c in ['history_segment', 'zip_code', 'channel', 'segment'] if c in df.columns]
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=axes[i], color='#F18F01')
    axes[i].set_title(f'{col} counts')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('count')
    axes[i].tick_params(axis='x', rotation=25)
for j in range(len(cat_cols), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb01_categorical_counts.png')
plt.show()

## 4) Treatment group analysis

In [ ]:
df['treatment_group'] = np.where(df['treatment'] == 1, 'Treated (Any Email)', 'Control (No Email)')
group_order = ['Control (No Email)', 'Treated (Any Email)']

counts = df['treatment_group'].value_counts().reindex(group_order).rename_axis('group').reset_index(name='n')
counts['share'] = counts['n'] / len(df)
display(counts)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=counts, x='group', y='n', ax=ax, palette='Set2')
ax.set_title('Treatment vs Control Counts')
ax.set_xlabel('Group')
ax.set_ylabel('Customers')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb01_treatment_control_counts.png')
plt.show()

In [ ]:
outcomes = ['visit', 'conversion', 'spend']
rates = df.groupby('treatment_group')[outcomes].mean().reindex(group_order)
display(rates)

for metric, title, fname in [
    ('visit', 'Visit Rate by Group', 'nb01_visit_rate_by_group.png'),
    ('conversion', 'Conversion Rate by Group', 'nb01_conversion_rate_by_group.png'),
    ('spend', 'Average Spend by Group', 'nb01_spend_by_group.png'),
]:
    chart = rates.reset_index()[['treatment_group', metric]]
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.barplot(data=chart, x='treatment_group', y=metric, ax=ax, palette='Set2')
    ax.set_title(title)
    ax.set_xlabel('Group')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    save_plot(fig, OUT_DIRS['figures'] / fname)
    plt.show()

Raw treatment-control differences above are descriptive signals only.
They do **not** reveal individual causal effect heterogeneity.

## 5) Covariate balance diagnostics (pre-treatment features only)
All covariates used for modeling are restricted to **pre-treatment** features only.
We exclude outcome columns (`visit`, `conversion`, `spend`) and treatment indicators from feature construction to avoid leakage.

In [ ]:
X_pre, feature_names = prepare_features(df, treatment_col='treatment')
treated_mask = df['treatment'] == 1
control_mask = df['treatment'] == 0

rows = []
for col in feature_names:
    smd = standardized_mean_difference(X_pre.loc[treated_mask, col], X_pre.loc[control_mask, col])
    rows.append({'feature': col, 'smd': smd, 'abs_smd': abs(smd)})

smd_df = pd.DataFrame(rows).sort_values('abs_smd', ascending=False).reset_index(drop=True)
display(smd_df.head(20))
smd_df.head(50).to_csv(OUT_DIRS['tables'] / 'nb01_smd_top50.csv', index=False)

In [ ]:
plot_df = smd_df.head(25).sort_values('smd')
fig, ax = plt.subplots(figsize=(12, 9))
sns.barplot(data=plot_df, x='smd', y='feature', palette='coolwarm', ax=ax)
ax.axvline(0.1, linestyle='--', color='black', linewidth=1)
ax.axvline(-0.1, linestyle='--', color='black', linewidth=1)
ax.set_title('Top 25 Standardized Mean Differences')
ax.set_xlabel('SMD')
ax.set_ylabel('Feature')
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb01_smd_balance_plot.png')
plt.show()

## 6) Causal framing
For each customer `i`:
- `Y_i(1)` = outcome if treated
- `Y_i(0)` = outcome if not treated
- `tau_i = Y_i(1) - Y_i(0)` is the individual treatment effect

We never observe both potential outcomes for the same individual.
Uplift models estimate this counterfactual difference for ranking decisions.

In [ ]:
demo = df[['treatment', 'conversion']].sample(8, random_state=SEED).reset_index(names='customer_id')
demo['observed_y'] = demo['conversion']
demo['Y(1)_potential'] = np.where(demo['treatment'] == 1, demo['observed_y'], np.nan)
demo['Y(0)_potential'] = np.where(demo['treatment'] == 0, demo['observed_y'], np.nan)
display(demo[['customer_id', 'treatment', 'observed_y', 'Y(1)_potential', 'Y(0)_potential']])

## 7) Marketing targeting framework
- **Persuadables**: convert if treated, not otherwise
- **Sure things**: convert regardless
- **Lost causes**: do not convert regardless
- **Sleeping dogs**: treatment harms conversion

Policy goal: prioritize persuadables while avoiding wasted or harmful targeting.

## 8) Business framing
Campaign decisions are budget-constrained:
- contact cost and channel fatigue exist
- not everyone should be targeted
- objective is incremental value, not maximum emails sent

## 9) Metric interpretation note
Qini and AUUC are useful **ranking** metrics for targeting quality.
They are not direct proof that treatment effects are perfectly estimated at the individual level.

## 10) End-of-notebook summary
- The Hillstrom dataset supports treatment-control uplift analysis at scale.
- The binary treatment simplification is explicit: any email vs no email.
- Covariate balance across pre-treatment features appears broadly reasonable.
- Naive conversion prediction is insufficient for causal targeting policy.
- Notebook 02 will compare naive treated-response ranking vs two-model uplift baseline on the same split.